# Phase 3: Banking Exploratory Data Analysis (EDA)

## Project: Credit Risk Modelling & Independent Model Validation (SR 11-7)

### Notebook Objectives
1. Analyze risk grade & sub-grade exposure distribution.
2. Evaluate observed default rates across risk grades A-G.
3. Assess borrower income, DTI, and FICO score risk trends.

In [1]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd

root_path = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = root_path / "src"
for p in [str(root_path), str(src_path)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("Environment & core risk libraries initialized successfully!")

Environment & core risk libraries initialized successfully!


In [2]:
# Data Loading Helper with Synthetic Fallback
data_file = root_path / "data" / "processed" / "accepted_2007_to_2018Q4_feature_engineered.csv.gz"
if data_file.is_file():
    df = pd.read_csv(data_file, nrows=50000, low_memory=False)
    bad = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off", "Late (31-120 days)"]
    good = ["Fully Paid", "Does not meet the credit policy. Status:Fully Paid"]
    df["target"] = np.nan
    df.loc[df["loan_status"].isin(bad), "target"] = 1.0
    df.loc[df["loan_status"].isin(good), "target"] = 0.0
    df = df.dropna(subset=["target"]).copy()
    df["target"] = df["target"].astype(int)
else:
    df = mock_df.copy()

print(f"Dataset Population Loaded: {len(df):,} loans | Default Rate: {df['target'].mean():.4%}")

Dataset Population Loaded: 44,252 loans | Default Rate: 20.9572%


In [3]:
grade_summary = df.groupby("grade", observed=False).agg(
    loan_count=("grade", "count"),
    total_exposure=("loan_amnt", "sum"),
    observed_default_rate=("target", "mean")
).reset_index()
grade_summary["observed_default_rate_pct"] = (grade_summary["observed_default_rate"] * 100).round(2)
grade_summary

,grade,loan_count,total_exposure,observed_default_rate,observed_default_rate_pct
0,A,8422,122248225.0,0.053788,5.38
1,B,13935,180335075.0,0.140653,14.07
2,C,12313,171978875.0,0.240478,24.05
3,D,5729,89538025.0,0.357654,35.77
4,E,2813,51361525.0,0.457163,45.72
5,F,847,16274475.0,0.537190,53.72
6,G,193,3353825.0,0.569948,56.99
